# **Project Brazilian e-commerce by Olist**

# Exploratory Data Analysis (EDA)

## Modeling Schema

<img src=..\docs\assets\data_schema.png width="900">


Source: https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce

## Data Dictionary

| Features | Description | Tables |
| :--- | :--- | :--- |
| customer_city | customer city name | customers |
| customer_id | key to the orders dataset. Each order has a unique customer_id. | customers, orders |
| customer_state | customer state | customers |
| customer_unique_id | unique identifier of a customer. | customers |
| customer_zip_code_prefix | first five digits of customer zip code | customers |
| freight_value | item freight value item (if an order has more than one item the freight value is splitted between items) | items |
| geolocation_city | city name | geolocation |
| geolocation_lat | latitude | geolocation |
| geolocation_lng | longitude | geolocation |
| geolocation_state | state | geolocation |
| geolocation_zip_code_prefix | first 5 digits of zip code | geolocation |
| order_approved_at | Shows the payment approval timestamp. | orders |
| order_delivered_carrier_date | Shows the order posting timestamp. When it was handled to the logistic partner. | orders |
| order_delivered_customer_date | Shows the actual order delivery date to the customer. | orders |
| order_estimated_delivery_date | Shows the estimated delivery date that was informed to customer at the purchase moment. | orders |
| order_id | order unique identifier | items, payments, reviews, orders |
| order_item_id | sequential number identifying number of items included in the same order | items |
| order_purchase_timestamp | Shows the purchase timestamp. | orders |
| order_status | Reference to the order status (delivered, shipped, etc). | orders |
| payment_installments | number of installments chosen by the customer. | payments |
| payment_sequential | a customer may pay an order with more than one payment method. If he does so, a sequence will be created to | payments |
| payment_type | method of payment chosen by the customer. | payments |
| payment_value | transaction value. | payments |
| price | item price | items |
| product_category_name | root category of product, in Portuguese. | products |
| > product_description_lenght | number of characters extracted from the product description. | products |
| product_height_cm | product height measured in centimeters. | products |
| product_id | product unique identifier | items, products |
| product_length_cm | product length measured in centimeters. | products |
| > product_name_lenght | number of characters extracted from the product name. | products |
| product_photos_qty | number of product published photos. | products |
| product_weight_g | product weight measured in grams. | products |
| product_width_cm | product width measured in centimeters. | products |
| review_answer_timestamp | Shows satisfaction survey answer timestamp. | reviews |
| review_comment_message | Comment message from the review left by the customer, in Portuguese. | reviews |
| review_comment_title | Comment title from the review left by the customer, in Portuguese. | reviews |
| review_creation_date | Shows the date in which the satisfaction survey was sent to the customer. | reviews |
| review_id | unique review identifier | reviews |
| review_score | Note ranging from 1 to 5 given by the customer on a satisfaction survey. | reviews |
| seller_city | seller city name | sellers |
| seller_id | seller unique identifier | items, sellers |
| seller_state | seller state | sellers |
| seller_zip_code_prefix | first 5 digits of seller zip code | sellers |
| shipping_limit_date | Shows the seller shipping limit date for handling the order over to the logistic partner. | items |


## Imports and configs

In [ ]:
# Roda o script de merge das tabela, e trata o problemas encontrados
# %run ../src/data_processing.py

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from src.eda_utils import *
from src.data_processing import *
from src.plots import plot_numeric_distribution, plot_categorical_counts

In [ ]:
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

In [ ]:
DATE_COLS = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
    "shipping_limit_date",
    "review_creation_date",
    "review_answer_timestamp",
]

CAT_COLS = [
    "customer_city",
    "customer_state",
    "order_status",
    "payment_types",
    "product_category_name",
    "product_category_name_english",
    "seller_city",
    "seller_state",
]

ORDER_LEVEL_COLS = [
    "customer_id",
    "order_status",
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
    "customer_unique_id",
    "customer_zip_code_prefix",
    "customer_city",
    "customer_state",
    "payment_value_total",
    "payment_installments_max",
    "payment_sequential_max",
    "payment_types",
    "n_payment_methods",
    "review_id",
    "review_score",
    "review_comment_title",
    "review_comment_message",
    "review_creation_date",
    "review_answer_timestamp",
    "customer_lat",
    "customer_lng",
    "customer_geo_n_points"
]

ITEM_LEVEL_COLS = [
    "order_item_id",
    "product_id",
    "seller_id",
    "shipping_limit_date",
    "price",
    "freight_value",
    "product_category_name",
    "product_name_length",
    "product_description_length",
    "product_photos_qty",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm",
    "product_category_name_english",
    "seller_zip_code_prefix",
    "seller_city",
    "seller_state",
    "seller_lat",
    "seller_lng",
    "seller_geo_n_points",
    "seller_customer_distance_km",
    "same_zip_prefix"
]

In [ ]:
pd.reset_option("display.max_rows")

## Reading Data

In [ ]:
customers = pd.read_csv("../data/raw/olist_customers_dataset.csv")
geolocation = pd.read_csv("../data/raw/olist_geolocation_dataset.csv")
order_items = pd.read_csv("../data/raw/olist_order_items_dataset.csv")
order_payments = pd.read_csv("../data/raw/olist_order_payments_dataset.csv")
order_reviews = pd.read_csv("../data/raw/olist_order_reviews_dataset.csv")
orders = pd.read_csv("../data/raw/olist_orders_dataset.csv")
products = pd.read_csv("../data/raw/olist_products_dataset.csv")
sellers = pd.read_csv("../data/raw/olist_sellers_dataset.csv")
category_translation = pd.read_csv("../data/raw/product_category_name_translation.csv")

In [ ]:
dfs = {
    "customers": customers,
    "geolocation": geolocation,
    "items": order_items,
    "payments": order_payments,
    "reviews": order_reviews,
    "orders": orders,
    "products": products,
    "sellers": sellers,
    "translation": category_translation,
}

## Initial EDA

### Summary Statistics

In [ ]:
for name, df in dfs.items():
    print(f"{name.upper()} {df.shape}:")
    display(dataframe_summary(df))

### Observations

- O order_id está duplicado em orders (deveria ser único)? `Não`
- review_id duplicado ou múltiplos review_id para o mesmo order_id em order_reviews? `Sim, será tratado no merge dos dados` 
- Unicidade de order_id + order_item_id em order_items? `Sim, não há duplicidade das chaves compostas`
- product_id duplicado em products, seller_id duplicado em sellers? `Não`
- review_score sempre entre 1 e 5 em order_reviews? `Sim`
- Valores negativos/zero em price, freight_value (order_items), payment_value (order_payments), product_weight_g (products)? `Não há valores negativos, porém as colunas freight_value, payment_installments, payment_value e product_weight_g estão com valor mínimo zero, o que pode fazer sentido no caso de freight_value, mas não para os demais.`

In [ ]:
dfs["payments"].payment_installments.unique()

In [ ]:
# Verifica se a combinação order_id + order_item_id é única
duplicates = dfs["items"].duplicated(subset=["order_id", "order_item_id"]).sum()
print(f"Linhas duplicadas na chave composta: {duplicates}")

In [ ]:
strange_values = {
    "items": ["freight_value"],
    "payments": ["payment_installments", "payment_value"],
    "products": ["product_weight_g"],
}

for table, columns in strange_values.items():
    for col in columns:
        mask = dfs[table][col] == 0
        print(f"coluna {col} zerada")
        display(dfs[table][mask])
        if table == "items":
            display(dfs[table][mask].value_counts("seller_id"))
            # display(show_multi_item_orders(dfs[table][mask], required_cols = dfs[table].columns))

- `freight_value` zerado pode ser considerado valor legitimo e por se tratar de vendedores especificos demonstra um padrão, vale a pena investigar mais sobre, principalmente após o merge das tabelas e checar se há algum padrão associado a categoria, localização e preço.
- `payment_installments` zerado é provavel erro, pois o número mínimo de parcelas de pagamento deveria ser 1, porém tratando-se apenas de dois pedidos tem impacto mínimo no projeto.
- `payment_value` zerado pode representar erro, principalmente para payment_type not_defined vale a pena verificar novamente após o merge.
- `product_weight_g` zerado é um valor impossivel, o que também indica erro.

O tratamento desses erros e dados nulos serão feitos após o merge das tabelas, pois com mais contexto é possivel tomar uma decisão mais acertiva com relação a descartar ou preencher os valores.

In [ ]:
mask = dfs["orders"]["order_delivered_customer_date"].isna()
dfs["orders"][mask].value_counts("order_status")

- A feature `order_delivered_customer_date` é essencial para o objetivo de prever possivel atraso e oder_status delivered é o que mais faz sentido neste contexto, então ter apenas 8 nulos nesta categoria é um bom sinal. 

### Cardinality Checking

1. `order_items` — esperado 1:N

In [ ]:
# Conta quantos itens (linhas) cada order_id tem
items_per_order = dfs["items"].groupby("order_id").size()

# Distribuição: quantos pedidos têm 1 item, 2 itens, 3 itens, etc.
print(items_per_order.value_counts().sort_index())

# Estatística resumida
print(items_per_order.describe())

2. `order_payments` — checar se existe 1:N

In [ ]:
# Conta quantas linhas de pagamento cada order_id tem
payments_per_order = dfs["payments"].groupby("order_id").size()

print(payments_per_order.value_counts().sort_index())
print(payments_per_order.describe())

# Opcional: olhar exemplos de pedidos com múltiplas linhas de pagamento
multi_payment_orders = payments_per_order[payments_per_order > 1].index
dfs["payments"][dfs["payments"]["order_id"].isin(multi_payment_orders)].sort_values(
    "order_id"
).head(20)

3. `order_reviews` — checar se é ~1:1

In [ ]:
reviews_per_order = dfs["reviews"].groupby("order_id")["review_id"].nunique()

multi_review_orders = (reviews_per_order > 1).sum()
print(
    f"Pedidos com múltiplos reviews: {multi_review_orders} ({multi_review_orders / reviews_per_order.shape[0]:.2%})"
)

multi_review_orders = reviews_per_order[reviews_per_order > 1].index
df_multi_review = dfs["reviews"][
    dfs["reviews"]["order_id"].isin(multi_review_orders)
].sort_values("order_id")

display(
    df_multi_review[
        [
            "order_id",
            "review_id",
            "review_score",
            "review_creation_date",
            "review_answer_timestamp",
        ]
    ].head(20)
)

- Deveria haver apenas um review por pedido, mas o usuário pode ter atualizada a sua review e gerado outra, a estratégia que faz mais sentido é manter apenas o ultimo review de acordo com a variável `review_answer_timestamp`.

## Merge

Para dar prosseguimento as analises irei juntar todas as tabelas, no entanto surgem algumas perguntas que devem ser respondidas antes do merge: 
- `"O que representa uma linha no meu dataset unificado?"` ->  após breve analise cheguei a conclusão que o nivel mais granular que ainda faz sentido é o de item do pedido (order_items), pois produto, vendedor, frete e preço variam por item e não por um pedido inteiro.
- `"Como as demais tabelas se relacionam com o item do pedido?"` -> Isso pode ser respondido pela relação de cardinalidade:

| Tabela | Relação com "item do pedido" |
|---|---|
| `orders` | 1 pedido → N itens (então 1:m em relação ao grão) |
| `order_items` | **é o próprio grão** |
| `products` | N itens podem repetir o mesmo produto (m:1) |
| `sellers` | N itens podem repetir o mesmo vendedor (m:1) |
| `customers` | 1 pedido = 1 cliente, mas N itens do mesmo pedido repetem o cliente (m:1 em relação ao grão) |
| `payments` | 1 pedido pode ter N pagamentos — **não bate direto com o grão "item"** |
| `reviews` | 1 pedido normalmente tem 1 review, raramente mais — **também não bate diretamente** |

payments e reviews são relações "por pedido", não "por item". Se eu simplesmente desse merge direto delas contra o dataset já expandido por item, qualquer pedido com 2 itens e 2 pagamentos viraria 4 linhas (2×2) — um produto cartesiano que não representa nada real, só um artefato do merge.

Dito isto, a ordem do merge fica:

```
orders → customers → order_items → products → sellers → payments (agregado) → reviews (dedup)
```

- **`orders + customers` primeiro**: relação 1:1 real (cada pedido tem exatamente um cliente), então é seguro e não altera a contagem de linhas. Fiz isso cedo porque é "barato" e não introduz risco.
- **`+ order_items`**: aqui, e só aqui, o dataset "explode" de fato — de propósito, porque é essa a granularidade que eu quero.
- **`+ products` e `+ sellers`**: ambos são m:1 em relação ao grão atual (vários itens podem apontar pro mesmo produto/vendedor), então o merge não altera a contagem de linhas — só adiciona colunas.
- **`+ payments`**: como *não* é m:1 em relação ao grão, eu não faço merge direto. Primeiro **agrego** (`groupby('order_id')`) para colapsar a granularidade de payments para "por pedido", e só depois faço o merge — agora sim é m:1 seguro.
- **`+ reviews`**: mesmo raciocínio — dedupliquei por `order_id` antes, transformando numa relação segura.



**ATENÇÃO: Todas as modificações na base de dados como merges, correções de erros e inserções também serão implementadas no arquivo de *data_processing.py* para que a pipeline de dados seja reprodutível.**

In [ ]:
tables = load_raw_tables()
olist_merged = build_merged_dataset(tables)
save_interim(olist_merged)

## Integrity Checks

In [ ]:
df_merged = pd.read_parquet("../data/interim/olist_merged.parquet")
df_merged.head(3)

In [ ]:
df_merged.shape

### Summary Statistics (item)

In [ ]:
dataframe_summary(df_merged)

### Observations (items)

Os dois objetivos principais do projeto estão relacionados ao grão no nivel de pedido, então será necessario agregar os dados por pedido, porém antes disto é necessario fazer correções e analises no nivel de item para não propagar os erros na hora de agregar os conjuntos de dados. Seguem algumas observações e problemas detecatados:

1. **`shipping_limit_date` com máximo em 2020-04-09.** O dataset cobre 2016-2018 (compras até 2018-09), então essa data está fora do período. Vale investigar as linhas afetadas.

2. **`product_weight_g` e `freight_value` com mínimo 0.** Como visto anteriormente peso zero é impossível e o frete pode ser válido.

3. 1.603 nulos idênticos em `product_category_name`, `product_name_length`, `product_description_length` e `product_photos_qty`, o que sugere as mesmas linhas com metadados ausentes.

4. Só 3 linhas sem pagamento (provavelmente um único pedido com 3 itens).

5. **`payment_installments_max` com mínimo 0.** Um pagamento com zero parcelas é suspeito.

6. **`payment_value_total` está inflado.** A média é 180,28, contra 140,64 de `price + freight_value` (120,65 + 19,99). Em pedidos com vários itens, cada linha repete o pagamento total do pedido, ou seja, essa coluna no grão de item não deve ser somada.

7. **`product_category_name_english` tem 1.627 nulos**, ou seja, **24 a mais** que a categoria em português. São categorias que não estão na tabela de tradução, e dá para traduzir manualmente.

**Distribuições**

- **Assimetria forte** em `price` (assimetria 7,9 e curtose 120, máximo 6.735 contra mediana de 74,99), em `payment_value_total` (assimetria 13,9) e em `product_weight_g`.

- **`review_score` concentrado em 5**: 56,5% das linhas (63.157 de 111.708), com assimetria de −1,27.

Obs: **112.650 linhas para 98.666 pedidos.** É o grão de item, como esperado. O `order_id` mais frequente aparece 21 vezes e `order_item_id` chega a 21, então há um pedido com 21 itens, que pesa 21 vezes mais que um pedido comum, ou seja, é necessario ter cautela ao colapsar para o nível de pedido. Outro ponto a resaltar é que tinhamos 99.441 pedidos na tabela **ORDERS**, porém perdemos 775 pedidos por conta da falta referência na tabela **ITEMS**.

#### shipping_limit_date

In [ ]:
df_merged.sort_values(by=["shipping_limit_date"], ascending=False)[DATE_COLS]

In [ ]:
for col in DATE_COLS:
    mask = df_merged[col].dt.year >= 2019
    print(f"{col} - {len(df_merged[mask])}")

- É um aparente erro, pois não faz sentido a compra ser aprovada em 2017 e ter prazo de entrega para 2020, irei inserir valor nulo visto que não temos uma data confiável.

In [ ]:
mask = df_merged["shipping_limit_date"].dt.year >= 2019
df_merged.loc[mask, "shipping_limit_date"] = pd.NaT
df_merged.loc[mask, DATE_COLS]

#### product_weight_g

In [ ]:
mask = df_merged["product_weight_g"] <= 0
df_merged.loc[mask]["product_id"].value_counts().to_frame()

- Só há 4 produtos com peso zerado

In [ ]:
df_merged.loc[mask, "product_weight_g"] = pd.NA

In [ ]:
df_merged["product_weight_g"].isna().sum()

#### freight_value

In [ ]:
mask = df_merged["freight_value"] == 0
for col in [
    "product_id",
    "product_category_name",
    "price",
    "seller_id",
    "seller_state",
    "seller_city",
]:
    display(df_merged[mask][col].value_counts()[:5].to_frame())

- Parece haver um padrão (mesmo vendedor, produto, categoria e estado) entre os itens de frete grátis, o que reforça a hipótese de valor legítmo e não um erro, por isso vamos manter como esta.

#### null values

In [ ]:
df_merged[["product_category_name", "product_name_length", "product_description_length", "product_photos_qty"]].isna().all(axis="columns").sum()

In [ ]:
df_merged[["product_weight_g", "product_width_cm", "product_length_cm", "product_height_cm"]].isna().all(axis="columns").sum()

In [ ]:
mask = df_merged[["payment_installments_max", "payment_types", "payment_value_total", "payment_sequential_max", "n_payment_methods"]].isna().all(axis="columns")
df_merged[mask]

In [ ]:
df_merged.loc[mask, "order_id"].value_counts().to_frame()

- De fato 1603 valores nulos pertencem as mesmas linhas nas quatro variáveis `product_category_name`, `product_name_length`, `product_description_length` e `product_photos_qty`, assim como os 18 valores nulos para `product_weight_g`, `product_width_cm`, `product_length_cm` e `product_height_cm`.
- Comprovado que 3 valores nulos relacionados as variáveis de pagamento pertecem realmente a um mesmo pedido

#### payment_installments_max

In [ ]:
mask = df_merged["payment_installments_max"] <= 0
df_merged[mask]

In [ ]:
df_merged[mask]["order_id"].value_counts().to_frame()

In [ ]:
df_merged.loc[mask, "payment_installments_max"] = pd.NA

#### Payment Consistency

In [ ]:
multi_item_df = show_multi_item_orders(df_merged)
multi_item_df

In [ ]:
consistency_df = check_payment_consistency(df_merged, tolerance=0.02)

# resumo rápido
display(consistency_df["is_consistent"].value_counts().to_frame())
display(f"Total de pedidos inconsistentes: {(~consistency_df['is_consistent']).sum()}")

# ver só os inconsistentes, ordenados pela maior diferença
inconsistent = consistency_df[~consistency_df["is_consistent"]].sort_values(
    "difference", ascending=False
)
inconsistent

In [ ]:
inconsistent.payment_types.value_counts()

- Apenas 278 pagamentos apresenta inconsistência considerando um tolerância de 2 centavos, isso significa que o merge com a tabela payments foi bem sucedido e os valores discrepantes podem ocorrer por sujeira ou outros fatores.  

#### product_category_name

In [ ]:
mask = (~df_merged['product_category_name'].isna() & df_merged['product_category_name_english'].isna())
df_merged.loc[mask, ['product_category_name', 'product_category_name_english']]

In [ ]:
mask = df_merged["product_category_name"] == "portateis_cozinha_e_preparadores_de_alimentos"
df_merged.loc[mask, 'product_category_name_english'] = "kitchen_portables_and_food_preparators"
mask = df_merged["product_category_name"] == "pc_gamer"
df_merged.loc[mask, 'product_category_name_english'] = "pc_gamer"

How many orders have multiple items?

In [ ]:
(df_merged["order_id"].value_counts() > 1).sum()

same_zip_prefix

In [ ]:
mask = (df_merged['seller_customer_distance_km'] == 0 )& (df_merged['same_zip_prefix'] != 1)
df_merged.loc[mask, ["same_zip_prefix", 'seller_customer_distance_km']]

In [ ]:
mask = df_merged['seller_customer_distance_km'] == 0
df_merged.loc[mask, ["same_zip_prefix", 'seller_customer_distance_km']]

- Infelizmente o zip_code_prefix tem apenas 5 digitos, ou seja, a área postal é mais abragente, o que faz alguns clientes e vendedores se localizarem na mesma área. 

### Order Aggregation 

In [ ]:
def check_order_level_consistency(df: pd.DataFrame, cols: list[str]) -> pd.Series:
    """Conta quantos pedidos têm mais de um valor distinto em cada coluna de pedido."""
    nunique = df.groupby("order_id")[cols].nunique(dropna=False)
    return (nunique > 1).sum().to_frame().rename({0: "nunique > 1"}, axis="columns")

print("Order level columns: ")
display(check_order_level_consistency(df_merged, ORDER_LEVEL_COLS))

print("Item level columns: ")
display(check_order_level_consistency(df_merged, ITEM_LEVEL_COLS))

- Nada fora do esperado, as variáveis do nivel de pedido tem apenas um único valor distinto quando agrupadas por pedido, enquanto variáveis do nivel de item podem ter mais de um. 

In [ ]:
# df_orders = collapse_item_level_to_order_level(
#     df_merged, order_level_cols = ORDER_LEVEL_COLUMNS
# )

# df_clean = clean_order_level_dataset(df_orders)
# save_processed(df_clean)

In [ ]:
df_processed = pd.read_parquet("../data/processed/olist_processed.parquet")
df_processed.head(3)

In [ ]:
columns = [
    "order_id",
    "order_status",
    "order_purchase_timestamp",
    "order_approved_at",
    "min_shipping_limit_date",
    "max_shipping_limit_date",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
    "review_creation_date",
    "review_answer_timestamp",
    "categories",
    "main_category",
    "n_categories",
    "mean_name_length",
    "mean_description_length",
    "min_photos_qty",
    "total_weight_g",
    "max_weight_g",
    "total_volume_cm3",
    "max_item_volume_cm3",
    "payment_installments_max",
    "payment_sequential_max",
    "n_payment_methods",
    "payment_types",
    "n_distinct_products",
    "n_items",
    "max_item_price",
    "mean_item_price",
    "total_price",
    "total_freight",
    "payment_value_total",
    "customer_id",
    "customer_unique_id",
    "customer_city",
    "customer_state",
    "customer_zip_code_prefix",
    "customer_lat",
    "customer_lng",
    "customer_geo_n_points",
    "n_sellers",
    "seller_states",
    "n_seller_states",
    "seller_cities",
    "max_seller_distance_km",
    "mean_seller_distance_km",
    "n_sellers_missing_distance",
    "min_seller_geo_n_points",
    "max_seller_geo_n_points",
    "review_id",
    "review_score",
    "review_comment_title",
    "review_comment_message",
]

In [ ]:
df_processed = df_processed[columns]
df_processed.shape

In [ ]:
df_processed.head(3)

### Summary Statistics (order)

In [ ]:
# save_table_as_image(dataframe_summary(df_processed), "df_processes_summary.png")

In [ ]:
# dataframe_summary(df_processed).to_excel("../reports/figures/df_processed_summary.xlsx")

In [ ]:
dataframe_summary(df_processed)

### Observations (orders)

Agora saimos de um dataset com 112.650 itens para um com 98.666 pedidos. Importante observar que temos apenas 9803 pedidos com mais de um item, então algumas colunas criadas que agregam informações do nivel de item, acabam tendo valores redundantes como os pares `avg_item_price` e `max_item_price`; `mean_seller_distance_km` e `max_seller_distance_km`; `categories` e `main_category`; `max_seller_geo_n_points` e `min_seller_geo_n_points`; `min_shipping_limit_date` e `max_shipping_limit_date`. 

**Consistente**

1. `order_id` e `customer_id` têm 98.666 únicos em 98.666 linhas.

2. Agora `total_price` + `total_freight` ≈ `payment_value_total` na média (137,75 + 22,82 ≈ 160,61) diferente de antes que estava inflado por conta da repetição dos valores no nivel de item.

3. As relações `n_seller_states` ≤ `n_sellers` ≤ `n_distinct_products` ≤ `n_items` se mantêm.

**Verificar**
1. `order_delivered_customer_date` tem 2.190 nulos é importante investigar se essas linhas não estão como `delivered`. As variáves `order_delivered_carrier_date` (1009) e `order_approved_at` (14) seguem a mesma lógica.

2. `customer_id` e `order_id` têm 98.666 únicos cada, e `customer_unique_id` tem 95.420. Ou seja, há pouca recompra, algo que pode aparecer na análise de negócio.

3. Colunas `review_*` com 749 nulos são pedidos sem avaliação.

4. `review_comment_title` e `review_comment_message` com respectivamente 88% e 59% de dados nulos, são campos opcionais. Provavelmente o mais útil é uma flag como `has_comment`.

**Distribuições**

- Como já visto antes a maioria das distribuições possuem assimetria forte a direita em especial `total_price` (assimetria 9,73 e curtose 266,07), `total_freight` (assimetria 12,05 e curtose 565,34) e `payment_value_total` (assimetria 9,23 e curtose 239,60), isso indica valores altos puxando a média para cima.


#### Payment consistency

In [ ]:
gap = (df_processed["payment_value_total"] - (df_processed["total_price"] + df_processed["total_freight"])).abs()
gap.mean()

#### Review nulls

In [ ]:
df_processed[
    ["review_comment_message", "review_creation_date", "review_id", "review_score"]
].isna().all(axis="columns").sum()

#### Dates

In [ ]:
date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
    "review_creation_date",
    "review_answer_timestamp",
    "max_shipping_limit_date",
    "min_shipping_limit_date"
]

#### NaT Values

In [ ]:
df_processed.order_status.value_counts().to_frame()

- **97,8% dos pedidos estão como `delivered`.** Para o modelo de atraso isso é otimo, pois só esses têm data de entrega para calcular o alvo.

In [ ]:
df_processed.order_status.value_counts().to_frame()

In [ ]:
df_processed.loc[df_processed["order_delivered_customer_date"].isna(), "order_status"].value_counts().to_frame()

In [ ]:
mask = df_processed.order_status != "delivered"
mask.sum()

Datas nulas de pedidos não entregues

In [ ]:
df_processed.loc[mask, date_cols].isna().sum().to_frame()

Datas nulas de pedidos entregues

In [ ]:
df_processed.loc[~mask, date_cols].isna().sum().to_frame()

- A variável `order_delivered_customer_date` tem apenas 8 nulos na categoria `delivered`, ou seja, 99,6% pertencem as demais categorias, o que é esperado, já que não há como um pedido ter data de entrega sem ter sido entregue. Em contrapartida as variáveis relacionadas a review tem comportamento inverso com 86,2% dos nulos na categoria `delivered`, porém considerando o somatório total de nulos em todas as variáveis de data temos ~2,6 vezes mais nulos nas demais categorias de `order_status`.  

#### Date Integrity

In [ ]:
date_order_rule = [
    ("order_purchase_timestamp", "order_approved_at"),
    ("order_purchase_timestamp", "order_delivered_carrier_date"),
    ("order_purchase_timestamp", "order_delivered_customer_date"),
    ("order_purchase_timestamp", "order_estimated_delivery_date"),
    ("order_purchase_timestamp", "max_shipping_limit_date"),
    ("order_approved_at", "order_delivered_carrier_date"),
    ("order_approved_at", "order_estimated_delivery_date"),
    ("order_approved_at", "order_delivered_customer_date"),
    ("order_delivered_carrier_date", "order_delivered_customer_date"),
]

In [ ]:
summary = check_date_integrity(df_processed, date_order_rule)
display(summary)

In [ ]:
# save_table_as_image(summary, "temporal_consistence.png", show_index=False,  max_col_width=6.0)

In [ ]:
mask = (
    df_processed["order_approved_at"].notna()
    & df_processed["order_delivered_customer_date"].notna()
) & (df_processed["order_approved_at"] >= df_processed["order_delivered_customer_date"])
df_processed[mask]["order_status"].value_counts()

In [ ]:
# Isola as linhas inconsistentes e calcula a magnitude da violação
violations = df_processed.loc[mask].copy()
violations["gap_hours"] = (
    violations["order_approved_at"] - violations["order_delivered_customer_date"]
).dt.total_seconds() / 3600

display(violations["gap_hours"].to_frame().describe())
display(violations["order_status"].value_counts().to_frame())

In [ ]:
df_eda = df_processed.loc[mask, :].copy()

days = df_eda["order_delivered_customer_date"] - df_eda["order_purchase_timestamp"]

days.dt.days.describe().to_frame()

- As inconsistências estão concentradas em `order_delivered_carrier_date` que apresenta problema em todas comparações, a hipótese mais plausível é que seja um campo ruidoso. Por outro lado em `order_approved_at` parece ser atraso no registro, pois cronologicamente um pedido só deveria ser enviado após a aprovação do pagamento e as violações vão contra essa lógica.
- O cutoff temporal do projeto é a variável `order_approved_at` o que evita vazamento de dados, isto é descrito no item 6 em `doc/business_understanding.md`, então as demais variáveis relacionadas ao periodo posterior não serão utilizadas como preditoras, o que torna a maioria das violações irrelevantes para o objetivo do projeto, contudo as violações `order_approved_at <= order_estimated_delivery_date` e `order_approved_at <= order_delivered_customer_date` irão prejudicar o calculo da variável alvo "delayed".
- Para o tratamento dos problemas apresentados decidi excluir nos casos de `order_approved_at <= order_estimated_delivery_date` e `order_approved_at <= order_delivered_customer_date` (approved <= customer, 61 linhas, e approved <= estimated, 7 linhas) as 68 linhas, cerca de 0,07% do total, então o impacto é desprezível, enquanto o restante optei por manter e só sinalizar criando uma flag booleana has_temporal_inconsistency.

Flag has_temporal_inconsistency

In [ ]:
has_temporal_inconsistency = pd.Series(False, index=df_processed.index)

for earlier, later in date_order_rule:
    mask = (df_processed[earlier].notna() & df_processed[later].notna()) & (
        df_processed[later] < df_processed[earlier]
    )
    has_temporal_inconsistency = has_temporal_inconsistency | mask

df_processed['has_temporal_inconsistency'] = has_temporal_inconsistency.astype("Int8")

In [ ]:
df_processed['has_temporal_inconsistency'].describe()

Remove date inconsistency (order_estimated_delivery_date and order_delivered_customer_date)

In [ ]:
date_order_rule = [
    ("order_approved_at", "order_estimated_delivery_date"),
    ("order_approved_at", "order_delivered_customer_date"),
]

for earlier, later in date_order_rule:
    mask = (df_processed[earlier].notna() & df_processed[later].notna()) & (
        df_processed[later] < df_processed[earlier]
    )
    df_processed.drop(df_processed[mask].index, inplace=True)
df_processed.reset_index(drop=True)

In [ ]:
delivered = df_processed[df_processed["order_status"] == "delivered"].copy()
delivered["month"] = delivered["order_purchase_timestamp"].dt.to_period("M")

delivered["is_late"] = (
    delivered["order_delivered_customer_date"]
    > delivered["order_estimated_delivery_date"]
)

monthly = delivered.groupby("month").agg(
    n_orders=("order_id", "size"),
    late_rate=("is_late", "mean"),
)

monthly

- 2016 é esparso, são 1, 265 e 1 pedidos em três meses. Os 100% e 0% de setembro e dezembro são apenas ruído onde n=1, ou seja, a melhor opção é iniciar o treino em 2017-01 e descartar 2016.
- A taxa de atraso não é estável no tempo, ela oscila de 3% a 21%, com picos em 2017-11 (14,3%, provavelmente Black Friday), 2018-02 (16,0%) e 2018-03 (21,4%). Isso tem consequências para o split temporal:

    - Se o teste for só agosto de 2018 (10,4%), a taxa-base é diferente da do treino, e métricas como accuracy ficam enganosas.
    - Os picos sugerem fatores sazonais ou operacionais que não aparecem nas features atuais. Vale incluir variáveis temporais (mês, dia da semana, proximidade de datas promocionais) e checar se o modelo as aproveita.
    - É bom considerar a validação com janelas móveis (por exemplo, TimeSeriesSplit) em vez de um único corte, para ver se o desempenho se mantém em regimes diferentes, como o de 2018-03.

### Delayed

In [ ]:
# Cria coluna com pedidos atrasados preservando os valores nulos
df_processed["delayed"] = (
    df_processed["order_delivered_customer_date"].dt.normalize()
    > df_processed["order_estimated_delivery_date"].dt.normalize()
).mask(df_processed["order_delivered_customer_date"].isna() | df_processed["order_estimated_delivery_date"].isna())


In [ ]:
df_processed["delayed"].sum()

In [ ]:
plot_categorical_counts(
    df_processed[df_processed["delayed"] == True], "order_status", multicolor=True
);

In [ ]:
df_processed.info()

In [ ]:
plot_categorical_counts(df_processed, "review_score", multicolor=True)

In [ ]:
df_merged["payment_types"].value_counts(dropna=False)

In [ ]:
df_merged.groupby("order_id")["review_id"].count().sort_values(ascending=True)